In [6]:
from groq import Groq
from dotenv import load_dotenv
from pydantic import BaseModel
from pypdf import PdfReader
from json import loads
import os

In [7]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("No Api Key Found")

client = Groq(api_key=api_key)
model = "openai/gpt-oss-20b"

Class & Schema design

In [8]:

class Job_D(BaseModel):
    role: str
    required_skills: list[str]
    preferred_skills: list[str]
    minimum_experience: float | None
    education_requirements: list[str]
    responsibilities: list[str]


job_d_schema = Job_D.model_json_schema()


Prompting

In [9]:

job_description = """
Role: Software Engineer Trainee
Position: MERN Stack Developer
Date of Joining: Immediate
Employment Type: Permanent
Experience Required: Fresher / Entry Level
About the Role
We are looking for a highly motivated and enthusiastic MERN Stack Developer who is passionate about building modern web applications.

The ideal candidate should have a solid understanding of web fundamentals, a keen interest in learning new technologies, and the ability to work independently as well as in a team.

This is a great opportunity for freshers who want to kickstart their career in full-stack development using MongoDB, Express.js, React.js, and Node.js — and grow into a complete product developer.

Key Responsibilities
Design, Develop, and maintenance of full-stack web applications using the MERN stack (MongoDB, Express, React, Node.js).
Write clean, efficient, and well-structured code following standard development
Learn and apply concepts of RESTful APIs, state management, and asynchronous programming.
Collaborate with team members to understand requirements, propose solutions, and deliver results within timelines.
Participate in code reviews and continuously improve code
Troubleshoot issues, debug applications, and contribute to performance
Stay updated with emerging web technologies and best
Take ownership of assigned modules or features with minimal
Demonstrate curiosity and eagerness to learn new tools, frameworks, and concepts
Required Skills & Knowledge
Technical Skills (Basic to Intermediate Knowledge Expected):

Good understanding of JavaScript (ES6+), HTML5, and CSS3.
Basic familiarity with js — components, props, state, hooks.
Understanding of js and Express.js for backend development.
Knowledge of MongoDB — schema design, CRUD operations,
Understanding of REST APIs and JSON-based
Basic knowledge of Git for version
Familiarity with package managers (npm/yarn) and Postman/Thunder Client for API
Additional Advantage (Good to Have):
Exposure to TypeScript, NestJs, js, or Redux.
Awareness of deployment processes (e.g., Vercel, Render, or AWS/Azure basics).
Basic understanding of UI/UX design principles. Soft Skills & Learning Mindset
Strong communication skills able to explain ideas clearly and collaborate
Demonstrated self-learning capability  ability to research and learn new tools or frameworks independently.
Problem-solving mindset with an analytical and logical
Highly self-motivated with a passion for technology and
Positive attitude, team player, and willingness to take ownership of assigned
Open to feedback and eager to continuously
Educational Qualification
Bachelor's degree in computer science, Information Technology, or any equivalent
Candidates with personal or academic projects in MERN stack will be given"""


system_prompt = f"""
You are an expert HR assistant.

Your job is to analyze job descriptions and extract
structured information from them.

Return ONLY valid JSON matching this schema:

{job_d_schema}
IMPORTANT:
Do NOT return the schema itself.
Do NOT return fields like "properties", "title" or "type".
Fill the schema with actual information extracted from the job description.

If minimum experience is not mentioned, return null.
If information for a list is missing, return an empty list.
Do not invent information.
"""

user_prompt = f"""
Analyze the following job description:

{job_description}
"""

Structuring

In [11]:
system_message = {
    "role": "system",
    "content": system_prompt
}

user_message = {
    "role": "user",
    "content": user_prompt
}

messages = [system_message, user_message]

Calling LLM to create Job Object

In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=messages,
    response_format={"type": "json_object"},
)

data = loads(response.choices[0].message.content)
job = Job_D(**data)
data